# Transformation 02 — Feature Pruning

Drop columns that are:
- **High-cardinality noise**: `DBA Name`, `AKA Name`, `Address` — free-text fields
  that are nearly unique per row; unlikely to generalize.
- **Constant / near-constant flags**: `flag_non_il_state`, `flag_longitude_outside_typical_range`.
- **Raw date columns**: after feature extraction, the original datetime columns are no longer needed.

In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from helpers.data_loader import DataLoader

train = pd.read_parquet(DataLoader.transformed('train_t01.parquet'))
val   = pd.read_parquet(DataLoader.transformed('val_t01.parquet'))
test  = pd.read_parquet(DataLoader.transformed('test_t01.parquet'))
print('Train shape BEFORE pruning:', train.shape)
print('Val   shape BEFORE pruning:', val.shape)
print('Test  shape BEFORE pruning:', test.shape)
print()
print('Columns:', list(train.columns))

## 1 · Identify columns to drop

In [ ]:
# High-cardinality noise columns
NOISE_COLS = ['DBA Name', 'AKA Name', 'Address']

# Constant / near-constant flag columns
FLAG_COLS = ['flag_non_il_state', 'flag_longitude_outside_typical_range', 'flag_non_chicago_city']

# Raw date columns (features have been extracted in previous step)
RAW_DATE_COLS = ['LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE']

# Combine all columns to drop
DROP_COLS = []
for col in NOISE_COLS + FLAG_COLS + RAW_DATE_COLS:
    if col in train.columns:
        DROP_COLS.append(col)
    else:
        print(f'Column "{col}" not found — skipping.')

print(f'Columns to drop ({len(DROP_COLS)}):', DROP_COLS)

## 2 · Quick check on constant flags

In [ ]:
for col in FLAG_COLS:
    if col in train.columns:
        vc = train[col].value_counts()
        print(f'{col}:')
        print(vc)
        print(f'  -> unique values: {train[col].nunique()}')
        print()

## 3 · Drop columns

In [ ]:
train = train.drop(columns=DROP_COLS)
val   = val.drop(columns=DROP_COLS)
test  = test.drop(columns=DROP_COLS)

print('Train shape AFTER pruning:', train.shape)
print('Val   shape AFTER pruning:', val.shape)
print('Test  shape AFTER pruning:', test.shape)
print()
print('Remaining columns:', list(train.columns))

In [ ]:
print(train['Facility Type'].nunique())
print(train['Facility Type'].value_counts().head(10))

## 4 · Save intermediate results

In [ ]:
train.to_parquet(DataLoader.transformed('train_t02.parquet'), index=False)
val.to_parquet(DataLoader.transformed('val_t02.parquet'),   index=False)
test.to_parquet(DataLoader.transformed('test_t02.parquet'),  index=False)
print('Saved train_t02.parquet, val_t02.parquet, and test_t02.parquet')

In [ ]:
train.columns.tolist()